# 🔗 KG Data Linking — GROQ API

Pipeline complet : Wikidata → génération de texte → annotation automatique



---
1. Installation & clé API
2. Prompt engineering
3. Test manuel sur quelques exemples
4. Pipeline en boucle sur tous les triplets
5. Annotation automatique
6. Visualisation & export

## 1. Installation

In [ ]:
!pip install -q openai
print(' openai installé')

 openai installé


## 2. Clé API GROQ


In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_KEY = userdata.get('GROQ_API_KEY')
    print(' Clé chargée depuis les Secrets Colab')
except Exception:
    from getpass import getpass
    GROQ_KEY = getpass('Colle ta clé Groc ici : ')
    print(' Clé saisie')

Colle ta clé Groc ici : ··········
 Clé saisie


## 3. Client GROQ

In [ ]:
import time, json, re
from dataclasses import dataclass, asdict
from openai import OpenAI

client = OpenAI(
    api_key  = GROQ_KEY,
    base_url = 'https://api.groq.com/openai/v1'
)
MODEL = 'llama-3.3-70b-versatile'

def call_llm(system_prompt, user_prompt, temperature=0.7):
    """Appel à l'API  via le SDK OpenAI."""
    t0 = time.time()
    response = client.chat.completions.create(
        model    = MODEL,
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
        max_tokens  = 600,
        temperature = temperature,
    )
    return {
        'text':       response.choices[0].message.content,
        'latency_ms': int((time.time() - t0) * 1000),
        'tokens':     response.usage.total_tokens,
    }

# Test de connexion
try:
    r = call_llm('You are helpful.', 'Reply with exactly: OK')
    print(f'GROQ API OK — modèle : {MODEL} — {r["latency_ms"]} ms')
    print(f'   Réponse : {r["text"].strip()}')
except Exception as e:
    print(f'❌ Erreur : {e}')

GROQ API OK — modèle : llama-3.3-70b-versatile — 660 ms
   Réponse : OK


## 4. Prompt Engineering

Trois templates selon le cas d'usage :
- **Single triple** : un texte par triplet (sujet, prédicat, objet)
- **Multi-triple** : centré sur une entité, couvre tous ses faits
- **Contextuel** : style contrôlé (encyclopédie, biographie, news)

In [ ]:
# ════════════════════════════════════════════════
# SYSTEM PROMPT
# ════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a knowledgeable writer producing natural, fluent encyclopedia-style text.
Your task: generate a short paragraph (2-4 sentences) that naturally incorporates knowledge graph entities.

Rules:
- Mention ALL provided entities naturally in flowing prose (no bullet points, no lists).
- Do NOT mention Wikidata or knowledge graphs.
- Be factually accurate and neutral.
- Entity names must appear EXACTLY as given (respect capitalization).
- After the paragraph, output a JSON block for annotation.

Output format:
TEXT: <your generated paragraph>
ANNOTATIONS: <JSON array: [{"entity_id":"Q...","label":"...","start":N,"end":N,"type":"..."}]>"""


# ════════════════════════════════════════════════
# PROMPT 1 : Single triple
# ════════════════════════════════════════════════
def build_single_prompt(triple):
    """
    Génère un prompt pour un seul triplet KG.
    Entrée : {'subject': {...}, 'predicate': {...}, 'object': {...}}
    """
    s, p, o = triple['subject'], triple['predicate'], triple['object']
    return (
        f'Generate a natural paragraph for this knowledge graph triple:\n'
        f'-Subject  : {s["label"]} (Wikidata ID: {s["id"]}, type: {s.get("type","entity")})\n'
        f'- Predicate: {p["label"]} (property: {p["id"]})\n'
        f'- Object   : {o["label"]} (Wikidata ID: {o["id"]}, type: {o.get("type","entity")})\n\n'
        f'Express naturally the relation: "{s["label"]}  {p["label"]}  {o["label"]}".'
    )


# ════════════════════════════════════════════════
# PROMPT 2 : Multi-triple (entity-centric)
# ════════════════════════════════════════════════
def build_multi_prompt(triples, subject_label=None):
    """
    Génère un prompt pour plusieurs triplets d'une même entité.
    Entrée : liste de triplets partageant le même sujet.
    """
    seen, entities, lines = set(), [], []
    for t in triples:
        for k in ('subject', 'object'):
            e = t[k]
            if e['id'] not in seen:
                seen.add(e['id'])
                entities.append(f'  - {e["label"]} ({e["id"]})')
        lines.append(
            f'  • {t["subject"]["label"]} —[{t["predicate"]["label"]}]→ {t["object"]["label"]}'
        )
    name = subject_label or triples[0]['subject']['label']
    return (
        f'Generate a paragraph about "{name}" using these knowledge graph triples:\n'
        + '\n'.join(lines)
        + '\n\nAll of these entities MUST appear naturally in the text:\n'
        + '\n'.join(entities)
    )


# ════════════════════════════════════════════════
# PROMPT 3 : Contextuel (style)
# ════════════════════════════════════════════════
def build_contextual_prompt(triple, style='encyclopedia'):
    """
    Génère un texte dans un style précis.
    style : 'encyclopedia' | 'biography' | 'news' | 'scientific'
    """
    styles = {
        'encyclopedia': 'a neutral, informative encyclopedia entry',
        'biography':    'a biographical introduction',
        'news':         'a brief news article excerpt',
        'scientific':   'a scientific abstract excerpt',
    }
    s, p, o = triple['subject'], triple['predicate'], triple['object']
    return (
        f'Write {styles.get(style, styles["encyclopedia"])} (2-3 sentences) '
        f'naturally mentioning:\n'
        f'Entity 1 : {s["label"]} (ID: {s["id"]}, type: {s.get("type","entity")})\n'
        f'Relation : {p["label"]} (property: {p["id"]})\n'
        f'Entity 2 : {o["label"]} (ID: {o["id"]}, type: {o.get("type","entity")})'
    )


# ════════════════════════════════════════════════
# PARSER de sortie
# ════════════════════════════════════════════════
def parse_output(raw):
    """Extrait le paragraphe et les annotations JSON de la réponse."""
    result = {'paragraph': raw.strip(), 'annotations': []}
    if 'TEXT:' in raw:
        after = raw.split('TEXT:', 1)[1]
        if 'ANNOTATIONS:' in after:
            para, ann = after.split('ANNOTATIONS:', 1)
            result['paragraph'] = para.strip()
            ann = ann.strip().replace('```json','').replace('```','').strip()
            try:    result['annotations'] = json.loads(ann)
            except: pass
        else:
            result['paragraph'] = after.strip()
    return result


print('** Prompts définis (single / multi / contextuel)')

# Aperçu d'un prompt
sample = {
    'subject':   {'id':'Q937',   'label':'Albert Einstein', 'type':'person'},
    'predicate': {'id':'P69',    'label':'educated at'},
    'object':    {'id':'Q11942', 'label':'ETH Zurich',       'type':'university'}
}
print('\nExemple de prompt single-triple :')

print(build_single_prompt(sample))

** Prompts définis (single / multi / contextuel)

Exemple de prompt single-triple :
Generate a natural paragraph for this knowledge graph triple:
-Subject  : Albert Einstein (Wikidata ID: Q937, type: person)
- Predicate: educated at (property: P69)
- Object   : ETH Zurich (Wikidata ID: Q11942, type: university)

Express naturally the relation: "Albert Einstein  educated at  ETH Zurich".


## 5. Sous-graphe Wikidata

In [ ]:
SUBGRAPH = {'triples': [
    {'subject':{'id':'Q937', 'label':'Albert Einstein','type':'person'},
     'predicate':{'id':'P69','label':'educated at'},
     'object':{'id':'Q11942','label':'ETH Zurich','type':'university'}},
    {'subject':{'id':'Q937','label':'Albert Einstein','type':'person'},
     'predicate':{'id':'P166','label':'award received'},
     'object':{'id':'Q38104','label':'Nobel Prize in Physics','type':'award'}},
    {'subject':{'id':'Q937','label':'Albert Einstein','type':'person'},
     'predicate':{'id':'P27','label':'country of citizenship'},
     'object':{'id':'Q183','label':'Germany','type':'country'}},
    {'subject':{'id':'Q7186','label':'Marie Curie','type':'person'},
     'predicate':{'id':'P166','label':'award received'},
     'object':{'id':'Q38104','label':'Nobel Prize in Physics','type':'award'}},
    {'subject':{'id':'Q7186','label':'Marie Curie','type':'person'},
     'predicate':{'id':'P19','label':'place of birth'},
     'object':{'id':'Q270','label':'Warsaw','type':'city'}},
    {'subject':{'id':'Q7186','label':'Marie Curie','type':'person'},
     'predicate':{'id':'P108','label':'employer'},
     'object':{'id':'Q209842','label':'University of Paris','type':'university'}},
    {'subject':{'id':'Q35637','label':'CERN','type':'organization'},
     'predicate':{'id':'P131','label':'located in'},
     'object':{'id':'Q39','label':'Switzerland','type':'country'}},
    {'subject':{'id':'Q2599','label':'Isaac Newton','type':'person'},
     'predicate':{'id':'P69','label':'educated at'},
     'object':{'id':'Q35794','label':'University of Cambridge','type':'university'}},
]}

print(f'✅ {len(SUBGRAPH["triples"])} triplets chargés\n')
for t in SUBGRAPH['triples']:
    print(f'  {t["subject"]["label"]:20s} {t["predicate"]["label"]:25s} {t["object"]["label"]}')  ##fait à la main

✅ 8 triplets chargés

  Albert Einstein      educated at               ETH Zurich
  Albert Einstein      award received            Nobel Prize in Physics
  Albert Einstein      country of citizenship    Germany
  Marie Curie          award received            Nobel Prize in Physics
  Marie Curie          place of birth            Warsaw
  Marie Curie          employer                  University of Paris
  CERN                 located in                Switzerland
  Isaac Newton         educated at               University of Cambridge


In [ ]:
import requests, time, json ## un peu automatique de wikidata

BASE_URL = "https://www.wikidata.org/w/api.php"
HEADERS  = {"User-Agent": "MonProjet/1.0 (contact@example.com)"}

# ── Propriétés cinéma pertinentes ────────────────────────────────────────────
CINEMA_PROPS = {
    "P57",   # réalisateur
    "P161",  # acteur
    "P162",  # producteur
    "P86",   # compositeur
    "P272",  # société de production
    "P136",  # genre
    "P495",  # pays d'origine
    "P364",  # langue originale
    "P179",  # série / franchise
    "P166",  # récompense reçue
    "P31",   # instance de
    "P577",  # date de publication  (valeur string, ignorée)
}

# ── Entités racines : films célèbres ─────────────────────────────────────────
CINEMA_SEEDS = [
    "Q190525",  # Le Parrain
    "Q44578",   # Titanic
    "Q172975",  # Pulp Fiction
    "Q83495",   # The Dark Knight
    "Q134773",  # Inception
]

# ── Types Wikidata → types lisibles pour le notebook ─────────────────────────
# (utilisés pour le champ "type" attendu par rule_annotate et render_html)
TYPE_MAP = {
    "Q5":         "person",        # être humain
    "Q11424":     "film",
    "Q36180":     "person",        # écrivain
    "Q2526255":   "person",        # réalisateur
    "Q10800557":  "person",        # acteur
    "Q222910":    "organization",  # société de production
    "Q11424":     "film",
    "Q201658":    "award",         # récompense cinéma
    "Q183":       "country",
    "Q6256":      "country",
}

# ── Fonctions utilitaires ────────────────────────────────────────────────────

def fetch_entities(ids: list[str]) -> dict:
    params = {
        "action":    "wbgetentities",
        "ids":       "|".join(ids[:50]),
        "languages": "fr|en",
        "format":    "json",
    }
    r = requests.get(BASE_URL, params=params, headers=HEADERS)
    r.raise_for_status()
    return r.json().get("entities", {})

def resolve_labels(ids: list[str]) -> dict:
    """Résout QIDs et PIDs → labels fr/en."""
    labels = {}
    unique = list(set(ids))
    for i in range(0, len(unique), 50):
        batch = unique[i:i+50]
        params = {
            "action":    "wbgetentities",
            "ids":       "|".join(batch),
            "props":     "labels",
            "languages": "fr|en",
            "format":    "json",
        }
        r = requests.get(BASE_URL, params=params, headers=HEADERS)
        for qid, ent in r.json().get("entities", {}).items():
            lbl = ent.get("labels", {})
            labels[qid] = (lbl.get("fr") or lbl.get("en") or {}).get("value", qid)
        time.sleep(0.1)
    return labels

def guess_type(entity: dict) -> str:
    """Devine le type d'une entité à partir de ses claims P31."""
    claims_p31 = entity.get("claims", {}).get("P31", [])
    for claim in claims_p31:
        val = (claim.get("mainsnak", {})
                    .get("datavalue", {})
                    .get("value", {})
                    .get("id", ""))
        if val in TYPE_MAP:
            return TYPE_MAP[val]
    # fallback sur la description
    desc = (entity.get("descriptions", {})
                  .get("en", {})
                  .get("value", "")).lower()
    if any(w in desc for w in ["actor", "actress", "director", "acteur", "réalisateur"]):
        return "person"
    if any(w in desc for w in ["film", "movie"]):
        return "film"
    if any(w in desc for w in ["company", "studio", "production"]):
        return "organization"
    if any(w in desc for w in ["award", "prize", "récompense"]):
        return "award"
    if any(w in desc for w in ["country", "pays"]):
        return "country"
    return "entity"

# ── Extraction du sous-graphe ────────────────────────────────────────────────

def build_cinema_subgraph(
    seed_ids:      list[str],
    allowed_props: set[str],
    hops:          int = 1,
    max_nodes:     int = 80,
) -> list[tuple]:
    """BFS filtré sur les propriétés cinéma."""
    visited, queue, triplets = set(), list(seed_ids), []

    for hop in range(hops):
        next_hop = []
        for i in range(0, len(queue), 50):
            if len(visited) >= max_nodes:
                print(f"  Limite {max_nodes} nœuds atteinte.")
                break
            batch = [e for e in queue[i:i+50] if e not in visited]
            if not batch:
                continue
            entities = fetch_entities(batch)
            visited.update(batch)
            for qid, entity in entities.items():
                if "missing" in entity:
                    continue
                for prop, claims in entity.get("claims", {}).items():
                    if prop not in allowed_props:
                        continue
                    for claim in claims:
                        snak = claim.get("mainsnak", {})
                        if snak.get("snaktype") != "value":
                            continue
                        dv = snak.get("datavalue", {})
                        if dv.get("type") == "wikibase-entityid":
                            obj = dv["value"]["id"]
                            triplets.append((qid, prop, obj))
                            if obj not in visited:
                                next_hop.append(obj)
            time.sleep(0.1)
        print(f"  Hop {hop+1} — {len(visited)} nœuds, {len(triplets)} triplets")
        queue = next_hop

    return triplets

# ── Exécution ────────────────────────────────────────────────────────────────

print("Extraction du sous-graphe cinéma...")
raw_triplets = build_cinema_subgraph(
    seed_ids      = CINEMA_SEEDS,
    allowed_props = CINEMA_PROPS,
    hops          = 1,
    max_nodes     = 80,
)
print(f"{len(raw_triplets)} triplets bruts extraits\n")

# ── Résolution des labels ────────────────────────────────────────────────────

all_qids = list({s for s,p,o in raw_triplets} | {o for s,p,o in raw_triplets})
all_pids = list({p for s,p,o in raw_triplets})

print("Résolution des labels d'entités...")
node_labels = resolve_labels(all_qids)

print("Résolution des labels de propriétés...")
prop_labels = resolve_labels(all_pids)

# ── Résolution des types ─────────────────────────────────────────────────────

print("Résolution des types...")
entity_types = {}
for i in range(0, len(all_qids), 50):
    batch = all_qids[i:i+50]
    entities = fetch_entities(batch)
    for qid, entity in entities.items():
        entity_types[qid] = guess_type(entity)
    time.sleep(0.1)

# ── Construction du SUBGRAPH au format attendu par le notebook ───────────────

SUBGRAPH = {"triples": []}

seen_triplets = set()
for s, p, o in raw_triplets:
    key = (s, p, o)
    if key in seen_triplets:
        continue
    seen_triplets.add(key)

    SUBGRAPH["triples"].append({
        "subject": {
            "id":    s,
            "label": node_labels.get(s, s),
            "type":  entity_types.get(s, "entity"),
        },
        "predicate": {
            "id":    p,
            "label": prop_labels.get(p, p),
        },
        "object": {
            "id":    o,
            "label": node_labels.get(o, o),
            "type":  entity_types.get(o, "entity"),
        },
    })

# ── Aperçu ───────────────────────────────────────────────────────────────────

print(f"\n{len(SUBGRAPH['triples'])} triplets dans SUBGRAPH\n")
for t in SUBGRAPH["triples"][:10]:
    print(f"  {t['subject']['label']:25s} [{t['predicate']['label']:20s}] {t['object']['label']}")

# ── Sauvegarde ───────────────────────────────────────────────────────────────

with open("subgraph_cinema.json", "w", encoding="utf-8") as f:
    json.dump(SUBGRAPH, f, ensure_ascii=False, indent=2)
print("\nSauvegardé dans subgraph_cinema.json")

Extraction du sous-graphe cinéma...
  Hop 1 — 5 nœuds, 372 triplets
372 triplets bruts extraits

Résolution des labels d'entités...
Résolution des labels de propriétés...
Résolution des types...

371 triplets dans SUBGRAPH

  Memento                   [nature de l’élément ] film
  Memento                   [distribution        ] Carrie-Anne Moss
  Memento                   [distribution        ] Guy Pearce
  Memento                   [distribution        ] Joe Pantoliano
  Memento                   [distribution        ] Mark Boone Junior
  Memento                   [distribution        ] Jorja Fox
  Memento                   [distribution        ] Stephen Tobolowsky
  Memento                   [distribution        ] Harriet Sansom Harris
  Memento                   [distribution        ] Callum Keith Rennie
  Memento                   [distribution        ] Larry Holden

Sauvegardé dans subgraph_cinema.json


## 6. Test manuel sur quelques exemples

Teste les 3 types de prompts avant de lancer le pipeline complet.

In [ ]:
# ── Test A : Single-triple (3 exemples) ─────────────────────────
print('='*60)
print('TEST A — Single-triple generation')
print('='*60)

for triple in SUBGRAPH['triples'][:3]:
    s, p, o = triple['subject'], triple['predicate'], triple['object']
    print(f'\n🔷 {s["label"]} {p["label"]} {o["label"]}')

    result = call_llm(SYSTEM_PROMPT, build_single_prompt(triple))
    parsed = parse_output(result['text'])

    print(f'   ⏱  {result["latency_ms"]} ms | 🪙 {result["tokens"]} tokens')
    print(f'   📝 {parsed["paragraph"]}')
    time.sleep(0.3)

TEST A — Single-triple generation

🔷 Albert Einstein educated at ETH Zurich
   ⏱  554 ms | 🪙 400 tokens
   📝 Albert Einstein, one of the most renowned physicists of the 20th century, was educated at ETH Zurich, a prestigious university in Switzerland. During his time at ETH Zurich, Einstein laid the foundation for his future academic pursuits, which would eventually lead to the development of the theory of relativity. ETH Zurich, with its strong emphasis on scientific research and innovation, provided Einstein with an ideal environment to explore his intellectual curiosity. This period of education played a significant role in shaping Einstein's intellectual trajectory.

ANNOTATIONS: [{"entity_id":"Q937","label":"Albert Einstein","start":0,"end":13,"type":"person"}, {"entity_id":"Q11942","label":"ETH Zurich","start":44,"end":53,"type":"university"}]

🔷 Albert Einstein award received Nobel Prize in Physics
   ⏱  1198 ms | 🪙 441 tokens
   📝 Albert Einstein, a renowned physicist, was hono

In [ ]:
# ── Test B : Multi-triple / entity-centric ───────────────────────
print('='*60)
print('TEST B — Entity-centric (Albert Einstein, 3 faits)')
print('='*60)

einstein = [t for t in SUBGRAPH['triples'] if t['subject']['id'] == 'Q937']
result   = call_llm(SYSTEM_PROMPT, build_multi_prompt(einstein, 'Albert Einstein'))
parsed   = parse_output(result['text'])

print(f'\n📝 {parsed["paragraph"]}')
print(f'\n✅ {result["latency_ms"]} ms | {result["tokens"]} tokens')

TEST B — Entity-centric (Albert Einstein, 3 faits)

📝 Albert Einstein, one of the most influential scientists of the 20th century, was educated at ETH Zurich, where he laid the foundation for his future groundbreaking work. Born in Germany, he would go on to make significant contributions to the field of physics, ultimately receiving the prestigious Nobel Prize in Physics for his remarkable achievements. Throughout his life, Einstein's work had a profound impact on our understanding of the universe, cementing his legacy as a pioneer in his field.

✅ 608 ms | 473 tokens


In [ ]:
# ── Test C : Variations de style ────────────────────────────────
print('='*60)
print('TEST C — Styles (encyclopédie / biographie / news)')
print('='*60)

base = SUBGRAPH['triples'][0]  # Einstein / ETH Zurich
for style in ['encyclopedia', 'biography', 'news']:
    result = call_llm(SYSTEM_PROMPT, build_contextual_prompt(base, style))
    parsed = parse_output(result['text'])
    print(f'\n🎨 [{style.upper()}]')
    print(f'   {parsed["paragraph"]}')
    time.sleep(0.3)

TEST C — Styles (encyclopédie / biographie / news)

🎨 [ENCYCLOPEDIA]
   Albert Einstein, a renowned physicist, was educated at ETH Zurich, where he laid the foundation for his future groundbreaking work. During his time at ETH Zurich, Einstein developed a deep understanding of physics and mathematics, which would later influence his theory of relativity. ETH Zurich, with its strong emphasis on scientific inquiry and academic excellence, played a significant role in shaping Einstein's intellectual trajectory.

ANNOTATIONS: [{"entity_id":"Q937","label":"Albert Einstein","start":0,"end":13,"type":"person"}, {"entity_id":"Q11942","label":"ETH Zurich","start":34,"end":43,"type":"university"}]

🎨 [BIOGRAPHY]
   Albert Einstein, one of the most influential scientists of the 20th century, was educated at ETH Zurich, a prestigious institution renowned for its academic excellence. During his time at ETH Zurich, Einstein laid the foundation for his future groundbreaking work, which would eventual

## 7. Annotateur automatique (rule-based)

In [ ]:
@dataclass
class EntitySpan:
    entity_id:    str
    label:        str
    entity_type:  str
    surface_form: str
    start:        int
    end:          int
    confidence:   float = 1.0
    def to_dict(self): return asdict(self)


def rule_annotate(text, entities):
    """
    Annotation automatique par matching exact + alias.
    - Matching exact du label
    - Pour les personnes : nom de famille seul
    - Pour les organisations : premier token
    """
    spans, seen = [], set()
    for e in entities:
        candidates = [e['label']]
        toks = e['label'].split()
        if e.get('type') == 'person' and len(toks) > 1:
            candidates += [toks[-1], toks[0]]          # nom et prénom seuls
        elif e.get('type') in ('university','organization') and len(toks) > 1:
            candidates.append(toks[0])                 # premier mot (ex: ETH)
        for surf in candidates:
            for m in re.finditer(r'\b' + re.escape(surf) + r'\b', text, re.IGNORECASE):
                k = (e['id'], m.start(), m.end())
                if k not in seen:
                    seen.add(k)
                    spans.append(EntitySpan(
                        entity_id    = e['id'],
                        label        = e['label'],
                        entity_type  = e.get('type','entity'),
                        surface_form = text[m.start():m.end()],
                        start        = m.start(),
                        end          = m.end(),
                        confidence   = 1.0 if text[m.start():m.end()] == e['label'] else 0.85
                    ))
    return sorted(spans, key=lambda s: s.start)


def format_annotated(text, spans):
    """Affiche le texte avec les entités annotées : [label|QID]"""
    result, cursor = [], 0
    for s in sorted(spans, key=lambda s: (s.start, -(s.end-s.start))):
        if s.start < cursor: continue
        result += [text[cursor:s.start], f'[{text[s.start:s.end]}|{s.entity_id}]']
        cursor = s.end
    return ''.join(result + [text[cursor:]])


def spans_to_iob(text, spans):
    """Convertit les spans en tags IOB2 (format CoNLL)."""
    toks = [(m.group(), m.start(), m.end()) for m in re.finditer(r'\S+', text)]
    tags = ['O'] * len(toks)
    for s in spans:
        for i, (tok, ts, te) in enumerate(toks):
            if ts >= s.start and te <= s.end:
                tags[i] = f"B-{s.entity_type.upper()}" if ts == s.start else f"I-{s.entity_type.upper()}"
    return [(tok, tag) for (tok,_,_), tag in zip(toks, tags)]


# Test de l'annotateur
test_txt  = 'Albert Einstein studied at ETH Zurich and received the Nobel Prize in Physics.'
test_ents = [
    {'id':'Q937',   'label':'Albert Einstein',       'type':'person'},
    {'id':'Q11942', 'label':'ETH Zurich',             'type':'university'},
    {'id':'Q38104', 'label':'Nobel Prize in Physics', 'type':'award'},
]
test_spans = rule_annotate(test_txt, test_ents)
print('✅ Annotateur OK :')
print(format_annotated(test_txt, test_spans))
print()
print('IOB2 :')
for tok, tag in spans_to_iob(test_txt, test_spans):
    if tag != 'O':
        print(f'  {tok:25s} {tag}')

✅ Annotateur OK :
[Albert Einstein|Q937] studied at [ETH Zurich|Q11942] and received the [Nobel Prize in Physics|Q38104].

IOB2 :
  Albert                    B-PERSON
  Einstein                  B-PERSON
  ETH                       B-UNIVERSITY
  Zurich                    I-UNIVERSITY
  Nobel                     B-AWARD
  Prize                     I-AWARD
  in                        I-AWARD


## 8. Pipeline complet — boucle sur tous les triplets

Deux modes disponibles :
- `single` : 1 texte par triplet
- `entity-centric` : 1 texte par entité sujet (regroupe tous ses faits)

In [ ]:
from collections import defaultdict

def run_pipeline(triples, mode='single', delay=0.5, verbose=True):
    """
    Boucle sur tous les triplets, génère le texte et annote.

    Args:
        triples : liste de triplets KG
        mode    : 'single' | 'entity-centric'
        delay   : secondes entre appels API
        verbose : afficher la progression

    Returns:
        corpus  : liste de records annotés
    """
    corpus = []

    if mode == 'entity-centric':
        groups = defaultdict(list)
        for t in triples:
            groups[t['subject']['id']].append(t)
        items = list(groups.items())
    else:
        items = list(enumerate(triples))

    total = len(items)

    for idx, (key, item) in enumerate(items, 1):
        try:
            # ── Prépare le prompt et les entités ──
            if mode == 'entity-centric':
                lbl    = item[0]['subject']['label']
                prompt = build_multi_prompt(item, lbl)
                seen, entities = set(), []
                for t in item:
                    for k in ('subject', 'object'):
                        e = t[k]
                        if e['id'] not in seen:
                            seen.add(e['id'])
                            entities.append({'id':e['id'],'label':e['label'],'type':e.get('type','entity')})
                record_id, triple_ref = f'entity_{idx:03d}', item
            else:
                t      = item
                prompt = build_single_prompt(t)
                entities = [
                    {'id':t['subject']['id'],'label':t['subject']['label'],'type':t['subject'].get('type','entity')},
                    {'id':t['object']['id'], 'label':t['object']['label'], 'type':t['object'].get('type','entity')},
                ]
                record_id, triple_ref = f'triple_{idx:03d}', t

            # ── Appel LLM ──
            result = call_llm(SYSTEM_PROMPT, prompt)
            parsed = parse_output(result['text'])

            # ── Annotation automatique ──
            spans  = rule_annotate(parsed['paragraph'], entities)

            corpus.append({
                'id':             record_id,
                'mode':           mode,
                'triple':         triple_ref,
                'paragraph':      parsed['paragraph'],
                'entities':       entities,
                'annotations':    [s.to_dict() for s in spans],
                'annotated_text': format_annotated(parsed['paragraph'], spans),
                'iob':            spans_to_iob(parsed['paragraph'], spans),
                'latency_ms':     result['latency_ms'],
                'tokens':         result['tokens'],
            })

            if verbose:
                lbl_str = (
                    item[0]['subject']['label'] if mode == 'entity-centric'
                    else f"{item['subject']['label']} → {item['object']['label']}"
                )
                print(f'[{idx:>2}/{total}] ✓ {lbl_str:45s} | {len(spans)} spans | {result["latency_ms"]}ms')

        except Exception as e:
            print(f'[{idx:>2}/{total}] ✗ Erreur : {e}')

        time.sleep(delay)

    return corpus

print('✅ Pipeline défini')

✅ Pipeline défini


In [ ]:
# ── Lancement ────────────────────────────────────────────────────
print('🚀 Pipeline en cours...\n' + '─'*60)

corpus = run_pipeline(
    triples = SUBGRAPH['triples'],
    mode    = 'single',    # ← change en 'entity-centric' si besoin
    delay   = 0.5,
)

print('─'*60)
total_spans  = sum(len(r['annotations']) for r in corpus)
total_tokens = sum(r['tokens'] for r in corpus)
print(f'\n✅ {len(corpus)} records | {total_spans} spans | {total_tokens} tokens')

🚀 Pipeline en cours...
────────────────────────────────────────────────────────────
[ 1/8] ✓ Albert Einstein → ETH Zurich                  | 17 spans | 693ms
[ 2/8] ✓ Albert Einstein → Nobel Prize in Physics      | 12 spans | 745ms
[ 3/8] ✓ Albert Einstein → Germany                     | 8 spans | 589ms
[ 4/8] ✓ Marie Curie → Nobel Prize in Physics          | 15 spans | 832ms
[ 5/8] ✓ Marie Curie → Warsaw                          | 16 spans | 873ms
[ 6/8] ✓ Marie Curie → University of Paris             | 22 spans | 592ms
[ 7/8] ✓ CERN → Switzerland                            | 5 spans | 615ms
[ 8/8] ✓ Isaac Newton → University of Cambridge        | 19 spans | 469ms
────────────────────────────────────────────────────────────

✅ 8 records | 114 spans | 3262 tokens


## 9. Visualisation des résultats

In [ ]:
from IPython.display import display, HTML
from collections import Counter

COLORS = {
    'person':       '#d4edda',
    'university':   '#cce5ff',
    'award':        '#fff3cd',
    'country':      '#f8d7da',
    'city':         '#e2d9f3',
    'organization': '#d1ecf1',
    'entity':       '#f0f0f0',
}

def render_html(record):
    text  = record['paragraph']
    spans = sorted(record['annotations'], key=lambda s:(s['start'],-(s['end']-s['start'])))
    result, cursor, seen = [], 0, set()
    for s in spans:
        if s['start'] < cursor: continue
        if (s['entity_id'], s['start']) in seen: continue
        seen.add((s['entity_id'], s['start']))
        result.append(text[cursor:s['start']])
        color = COLORS.get(s['entity_type'], '#f0f0f0')
        result.append(
            f'<mark style="background:{color};border-radius:3px;padding:1px 5px;font-weight:600"'
            f' title="{s["entity_id"]} ({s["entity_type"]})">'
            f'{text[s["start"]:s["end"]]}'
            f'<sup style="font-size:.65em;color:#555">{s["entity_id"]}</sup></mark>'
        )
        cursor = s['end']
    result.append(text[cursor:])
    return ''.join(result)

parts = ['<style>div.r{margin:8px 0;padding:12px;border:1px solid #ddd;border-radius:8px;font-family:sans-serif;line-height:1.7}</style>']
for r in corpus:
    t = r['triple']
    title = (
        t[0]['subject']['label'] if isinstance(t, list)
        else f"{t['subject']['label']} —[{t['predicate']['label']}]→ {t['object']['label']}"
    )
    parts.append(
        f'<div class="r">'
        f'<div style="font-size:.78em;color:#888;margin-bottom:4px">🔷 {title}</div>'
        f'<div>{render_html(r)}</div>'
        f'<div style="font-size:.72em;color:#aaa;margin-top:4px">'
        f'⏱ {r["latency_ms"]}ms · 🪙 {r["tokens"]} tokens · 🏷 {len(r["annotations"])} spans'
        f'</div></div>'
    )
display(HTML('\n'.join(parts)))

# Stats
tc = Counter(s['entity_type'] for r in corpus for s in r['annotations'])
print('\n📊 Types d\'entités annotées :')
for et, n in tc.most_common():
    print(f'  {et:15s} {"█"*n} {n}')


📊 Types d'entités annotées :
  person          ███████████████████████████████████████████████████████████████████ 67
  university      █████████████████████████████ 29
  award           ██████ 6
  country         █████ 5
  city            ████ 4
  organization    ███ 3


## 10. Export JSON + CoNLL

In [ ]:
import os
from google.colab import files
os.makedirs('/content/output', exist_ok=True)

# JSON
json_path = '/content/output/corpus.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({'model': MODEL, 'corpus': corpus}, f, ensure_ascii=False, indent=2)

# CoNLL IOB2
conll_path = '/content/output/corpus.conll'
with open(conll_path, 'w', encoding='utf-8') as f:
    for r in corpus:
        f.write(f'# {r["id"]}\n')
        for tok, tag in r['iob']:
            f.write(f'{tok}\t{tag}\n')
        f.write('\n')

print(f'✅ corpus.json  ({os.path.getsize(json_path)//1024} Ko)')
print(f'✅ corpus.conll ({os.path.getsize(conll_path)} octets)')
files.download(json_path)
files.download(conll_path)

✅ corpus.json  (85 Ko)
✅ corpus.conll (7838 octets)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>